# Stage 6: Model Building & Training

## Three Models:
1. **ResNet-50** (fine-tuned) — supervised classifier
2. **Custom CNN** — built from scratch
3. **Convolutional Autoencoder** — unsupervised anomaly detection

In [18]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SAVE_DIR = Path('../src/saved_models')
SAVE_DIR.mkdir(parents=True, exist_ok=True)
print(f'Device: {DEVICE}')

Device: cpu


In [19]:
# ============================================================
# MODEL 1: Fine-tuned ResNet-50
# ============================================================
class ResNet50Classifier(nn.Module):
    """
    ResNet-50 fine-tuned for binary defect classification.
    Strategy: freeze early layers, fine-tune layer4 + classifier.
    """
    def __init__(self, num_classes=2, pretrained=True, freeze_backbone=True):
        super().__init__()
        weights = 'IMAGENET1K_V2' if pretrained else None
        self.backbone = models.resnet50(weights=weights)
        
        if freeze_backbone:
            # Freeze all layers except layer4 and fc
            for name, param in self.backbone.named_parameters():
                if 'layer4' not in name and 'fc' not in name:
                    param.requires_grad = False
        
        # Replace final layer
        in_features = self.backbone.fc.in_features  # 2048
        self.backbone.fc = nn.Sequential(
            nn.Dropout(p=0.5),
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(512, num_classes)
        )
    
    def forward(self, x):
        return self.backbone(x)


model_resnet = ResNet50Classifier(num_classes=2).to(DEVICE)

# Count trainable parameters
trainable = sum(p.numel() for p in model_resnet.parameters() if p.requires_grad)
total = sum(p.numel() for p in model_resnet.parameters())
print(f'ResNet-50: {trainable:,} trainable / {total:,} total parameters')

ResNet-50: 16,014,850 trainable / 24,558,146 total parameters


In [20]:
# ============================================================
# MODEL 2: Custom CNN from Scratch
# ============================================================
class CustomCNN(nn.Module):
    """
    Lightweight CNN built from scratch for defect classification.
    Architecture: 4 conv blocks + global avg pool + classifier
    """
    def __init__(self, num_classes=2):
        super().__init__()
        
        def conv_block(in_ch, out_ch, stride=1):
            return nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True),
                nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2, 2)
            )
        
        self.features = nn.Sequential(
            conv_block(3, 32),    # 224 -> 112
            conv_block(32, 64),   # 112 -> 56
            conv_block(64, 128),  # 56 -> 28
            conv_block(128, 256), # 28 -> 14
        )
        self.pool = nn.AdaptiveAvgPool2d((1, 1))  # -> (B, 256, 1, 1)
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )
    
    def forward(self, x):
        x = self.features(x)
        x = self.pool(x).squeeze(-1).squeeze(-1)
        return self.classifier(x)


model_cnn = CustomCNN(num_classes=2).to(DEVICE)
params_cnn = sum(p.numel() for p in model_cnn.parameters())
print(f'Custom CNN: {params_cnn:,} parameters')

Custom CNN: 1,206,370 parameters


In [21]:
# ============================================================
# MODEL 3: Convolutional Autoencoder
# ============================================================
class ConvAutoencoder(nn.Module):
    """
    Convolutional Autoencoder for anomaly detection.
    Trained ONLY on normal images.
    Anomaly score = reconstruction error (MSE or SSIM).
    Defective images → high reconstruction error.
    """
    def __init__(self, latent_dim=512):
        super().__init__()
        
        # Encoder: 224x224 -> bottleneck
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 4, stride=2, padding=1),   # -> 112
            nn.ReLU(),
            nn.Conv2d(32, 64, 4, stride=2, padding=1),  # -> 56
            nn.ReLU(),
            nn.Conv2d(64, 128, 4, stride=2, padding=1), # -> 28
            nn.ReLU(),
            nn.Conv2d(128, 256, 4, stride=2, padding=1),# -> 14
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(256 * 14 * 14, latent_dim),
            nn.ReLU()
        )
        
        # Decoder: bottleneck -> 224x224
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 256 * 14 * 14),
            nn.ReLU(),
            nn.Unflatten(1, (256, 14, 14)),
            nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1), # -> 28
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1),  # -> 56
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1),   # -> 112
            nn.ReLU(),
            nn.ConvTranspose2d(32, 3, 4, stride=2, padding=1),    # -> 224
            nn.Sigmoid()
        )
    
    def forward(self, x):
        latent = self.encoder(x)
        reconstruction = self.decoder(latent)
        return reconstruction, latent


autoencoder = ConvAutoencoder(latent_dim=512).to(DEVICE)
params_ae = sum(p.numel() for p in autoencoder.parameters())
print(f'Autoencoder: {params_ae:,} parameters')

Autoencoder: 52,810,947 parameters


In [22]:
# ============================================================
# Training Function (generic — works for ResNet and CNN)
# ============================================================
def train_classifier(model, train_loader, val_loader, epochs=20,
                     lr=1e-4, class_weights=None, model_name='model'):
    """
    Train a binary CNN classifier with early stopping.
    """
    if class_weights is not None:
        criterion = nn.CrossEntropyLoss(weight=class_weights.to(DEVICE))
    else:
        criterion = nn.CrossEntropyLoss()
    
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=3, factor=0.5)
    
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_val_loss = float('inf')
    patience_counter = 0
    EARLY_STOP = 5
    
    for epoch in range(epochs):
        # --- TRAIN ---
        model.train()
        train_loss, train_correct = 0, 0
        for imgs, labels in tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs} [Train]', leave=False):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * imgs.size(0)
            train_correct += (outputs.argmax(1) == labels).sum().item()
        
        # --- VALIDATE ---
        model.eval()
        val_loss, val_correct = 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                outputs = model(imgs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * imgs.size(0)
                val_correct += (outputs.argmax(1) == labels).sum().item()
        
        n_train = len(train_loader.dataset)
        n_val = len(val_loader.dataset)
        tl = train_loss / n_train
        vl = val_loss / n_val
        ta = train_correct / n_train
        va = val_correct / n_val
        
        history['train_loss'].append(tl)
        history['val_loss'].append(vl)
        history['train_acc'].append(ta)
        history['val_acc'].append(va)
        
        print(f'Epoch {epoch+1:3d}/{epochs} | Loss: {tl:.4f}/{vl:.4f} | Acc: {ta:.4f}/{va:.4f}')
        
        scheduler.step(vl)
        
        if vl < best_val_loss:
            best_val_loss = vl
            patience_counter = 0
            torch.save(model.state_dict(), SAVE_DIR / f'{model_name}_best.pth')
            print(f'  ✓ Model saved (val_loss={vl:.4f})')
        else:
            patience_counter += 1
            if patience_counter >= EARLY_STOP:
                print(f'Early stopping at epoch {epoch+1}')
                break
    
    return history


def plot_training_history(history, model_name):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    
    ax1.plot(history['train_loss'], label='Train')
    ax1.plot(history['val_loss'], label='Validation')
    ax1.set_title(f'{model_name} — Loss')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
    ax1.legend()
    
    ax2.plot(history['train_acc'], label='Train')
    ax2.plot(history['val_acc'], label='Validation')
    ax2.set_title(f'{model_name} — Accuracy')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
    ax2.legend()
    
    plt.tight_layout()
    plt.savefig(f'../assets/training_{model_name}.png', dpi=150)
    plt.show()


print('Training functions defined.')
print('Run: history = train_classifier(model_resnet, train_loader, val_loader, epochs=20)')

Training functions defined.
Run: history = train_classifier(model_resnet, train_loader, val_loader, epochs=20)


In [23]:
# ============================================================
# Autoencoder Training (trained only on normal images)
# ============================================================
def train_autoencoder(ae, normal_loader, epochs=30, lr=1e-3):
    optimizer = optim.Adam(ae.parameters(), lr=lr)
    criterion = nn.MSELoss()
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    losses = []
    
    for epoch in range(epochs):
        ae.train()
        epoch_loss = 0
        for imgs, _ in tqdm(normal_loader, desc=f'AE Epoch {epoch+1}/{epochs}', leave=False):
            imgs = imgs.to(DEVICE)
            optimizer.zero_grad()
            recon, _ = ae(imgs)
            loss = criterion(recon, imgs)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * imgs.size(0)
        
        avg_loss = epoch_loss / len(normal_loader.dataset)
        losses.append(avg_loss)
        scheduler.step()
        if (epoch + 1) % 5 == 0:
            print(f'AE Epoch {epoch+1}/{epochs} | MSE Loss: {avg_loss:.6f}')
    
    torch.save(ae.state_dict(), SAVE_DIR / 'autoencoder_best.pth')
    return losses


def compute_anomaly_scores(ae, loader):
    """Compute reconstruction error for each image (anomaly score)."""
    ae.eval()
    scores, labels = [], []
    criterion = nn.MSELoss(reduction='none')
    with torch.no_grad():
        for imgs, lbls in loader:
            imgs = imgs.to(DEVICE)
            recon, _ = ae(imgs)
            # Per-image MSE
            mse = criterion(recon, imgs).mean(dim=[1, 2, 3])
            scores.extend(mse.cpu().numpy())
            labels.extend(lbls.numpy())
    return np.array(scores), np.array(labels)


print('Autoencoder training functions defined.')

Autoencoder training functions defined.


In [ ]:
# ============================================================
# COMPLETE DATASET + DATALOADER SETUP
# Copy-paste this BEFORE calling train_classifier(...)
# ============================================================

from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split, Subset
import torch
import numpy as np
import os
from pathlib import Path

# ============================================================
# STEP 1: DATASET PATH
# CHANGE THIS to your actual dataset folder
# Example structure:
# dataset/
#   ├── good/
#   └── defective/
# ============================================================
DATASET_DIR = r"C:\Users\LENOVO\OneDrive\Desktop\defect-detection-project\data"   # <-- CHANGE IF NEEDED

if not os.path.exists(DATASET_DIR):
    raise FileNotFoundError(f"Dataset folder not found: {DATASET_DIR}")

print("Dataset path found:", DATASET_DIR)
print("Folders:", os.listdir(DATASET_DIR))


# ============================================================
# STEP 2: IMAGE TRANSFORMS
# ============================================================
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])


# ============================================================
# STEP 3: LOAD DATASET
# ============================================================
full_dataset = datasets.ImageFolder(DATASET_DIR, transform=transform)

print("\n========== DATASET INFO ==========")
print("Classes:", full_dataset.classes)
print("Class mapping:", full_dataset.class_to_idx)
print("Total images:", len(full_dataset))


# ============================================================
# STEP 4: TRAIN / VALIDATION SPLIT
# ============================================================
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = random_split(
    full_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

print("\n========== SPLIT INFO ==========")
print("Train images:", len(train_dataset))
print("Validation images:", len(val_dataset))


# ============================================================
# STEP 5: DATALOADERS
# ============================================================
BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


# ============================================================
# STEP 6: CLASS WEIGHTS (for imbalance)
# ============================================================
train_labels = [full_dataset.targets[i] for i in train_dataset.indices]

class_counts = np.bincount(train_labels)

class_weights = torch.tensor(
    [len(train_labels) / (len(class_counts) * c) for c in class_counts],
    dtype=torch.float32
)

print("\n========== CLASS INFO ==========")
print("Class counts:", class_counts)
print("Class weights:", class_weights)


# ============================================================
# STEP 7: NORMAL-ONLY DATASET FOR AUTOENCODER
# ============================================================
# Automatically detect "good" class
if 'good' in full_dataset.class_to_idx:
    normal_class = full_dataset.class_to_idx['good']
elif 'normal' in full_dataset.class_to_idx:
    normal_class = full_dataset.class_to_idx['normal']
else:
    # fallback: assume first class is normal
    normal_class = 0

normal_indices = [
    i for i, label in enumerate(full_dataset.targets)
    if label == normal_class
]

normal_dataset = Subset(full_dataset, normal_indices)

normal_loader = DataLoader(
    normal_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

print("\n========== AUTOENCODER INFO ==========")
print("Normal class index:", normal_class)
print("Normal-only images:", len(normal_dataset))


# ============================================================
# STEP 8: FINAL CHECK
# ============================================================
print("\n========== LOADER CHECK ==========")
print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Normal batches:", len(normal_loader))


# ============================================================
# STEP 9: OPTIONAL SAMPLE TEST
# ============================================================
sample_imgs, sample_labels = next(iter(train_loader))

print("\n========== SAMPLE BATCH ==========")
print("Image batch shape:", sample_imgs.shape)   # Expected: [B,3,224,224]
print("Label batch shape:", sample_labels.shape)


# ============================================================
# STEP 10: TRAIN MODELS
# Uncomment and run one by one
# ============================================================

# ---------------- RESNET-50 ----------------
history_resnet = train_classifier(
    model_resnet,
    train_loader,
    val_loader,
    epochs=20,
    lr=1e-4,
    class_weights=class_weights,
    model_name='resnet50'
)

# ---------------- CUSTOM CNN ----------------
history_cnn = train_classifier(
    model_cnn,
    train_loader,
    val_loader,
    epochs=20,
    lr=1e-3,
    class_weights=class_weights,
    model_name='customcnn'
)

# ---------------- AUTOENCODER ----------------
ae_losses = train_autoencoder(
    autoencoder,
    normal_loader,
    epochs=30,
    lr=1e-3
)


# ============================================================
# STEP 11: VERIFY SAVED MODELS
# ============================================================
print("\n========== SAVED MODELS ==========")
if SAVE_DIR.exists():
    for file in SAVE_DIR.iterdir():
        print(file.name)
else:
    print("SAVE_DIR not found.")


# ============================================================
# STEP 12: COMPUTE AUTOENCODER THRESHOLD
# ============================================================
scores, labels = compute_anomaly_scores(autoencoder, val_loader)

normal_scores = scores[labels == normal_class]

AE_THRESHOLD = normal_scores.mean() + 3 * normal_scores.std()

print("\n========== AUTOENCODER THRESHOLD ==========")
print("AE_THRESHOLD =", AE_THRESHOLD)


# ============================================================
# STEP 13: SAVE THRESHOLD
# ============================================================
with open(SAVE_DIR / "ae_threshold.txt", "w") as f:
    f.write(str(AE_THRESHOLD))

print("AE threshold saved.")

FileNotFoundError: Dataset folder not found: /content/dataset

In [25]:
from pathlib import Path

print("SAVE_DIR =", SAVE_DIR.resolve())
print("Exists:", SAVE_DIR.exists())
print("Contents:", list(SAVE_DIR.iterdir()))

SAVE_DIR = C:\Users\LENOVO\OneDrive\Desktop\defect-detection-project\src\saved_models
Exists: True
Contents: []


In [24]:
import os
print(os.listdir("../src/saved_models"))

[]
